In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
df_features = pd.read_csv("../data/processed/cleaned_waitlist.csv")
df_features.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,INIT_DATE,...,DIAG_KI,MULTIORG,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event,DIALYSIS_DURATION,DIALYSIS_AFTER_LISTING
0,Y,-1.0,F,B,31.63,2080.0,4099,0.0,53,2020-03-25,...,-1.0,N,13609,died,1,0,0,287.0,726.0,0
1,Y,-1.0,M,A,30.04,2070.0,4099,0.0,56,2020-02-14,...,-1.0,N,6975,removed_administrative,0,0,0,2322.0,912.0,0
2,N,-1.0,F,O,32.85,2070.0,4099,0.0,47,2020-05-27,...,-1.0,N,19716,died,1,0,0,604.0,0.0,0
3,Y,-1.0,M,A,20.00,2090.0,4099,0.0,61,2020-04-02,...,-1.0,N,8587,removed_too_sick,1,0,0,909.0,453.0,0
4,Y,-1.0,M,AB,23.30,2070.0,4010,0.0,61,2020-02-05,...,-1.0,N,18352,removed_too_sick,1,0,0,231.0,391.0,0


In [3]:
df_features['FUNC_STAT_TCR'].value_counts()

FUNC_STAT_TCR
 2080.0    139465
 2070.0    111480
 2090.0    109306
 2060.0     36961
 2100.0     35402
 2050.0     23920
 998.0       8697
 2040.0      8364
 4100.0      4642
-1.0         3974
 2020.0      2936
 4080.0      2812
 4090.0      2413
 2030.0      1905
 4070.0      1029
 4060.0       505
 2010.0       371
 4040.0       187
 4050.0       157
 996.0        121
 4030.0        89
 4010.0        43
 4020.0        23
 1.0            1
Name: count, dtype: int64

In [4]:
# FUNC_STAT_TCR mixes two performance scales plus three non-scale codes.
# Decode per data_dictionary.md: thousands digit is the scale, last three digits
# are the percentage. 996/998/1 are categorical, not scores.
scale_digit = df_features['FUNC_STAT_TCR'] // 1000 #integer math to find out if we have a Karnofsky or Lansky

conditions = [
    scale_digit == 2,
    scale_digit == 4,
    df_features['FUNC_STAT_TCR'] == 996,
    df_features['FUNC_STAT_TCR'] == 998,
    df_features['FUNC_STAT_TCR'] == 1,
]

choices = [
    'karnofsky',
    'lansky',
    'not_applicable_infant',
    'unknown',
    'adl_independent',
]

# nulls fall through to default and are treated as unknown
df_features['functional_scale'] = np.select(conditions, choices, default='unknown')

df_features['functional_scale'].value_counts()

functional_scale
karnofsky                470110
unknown                   12671
lansky                    11900
not_applicable_infant       121
adl_independent               1
Name: count, dtype: int64

In [5]:
is_scale = (scale_digit == 2) | (scale_digit == 4)
df_features['functional_percent'] = np.where(is_scale, df_features['FUNC_STAT_TCR'] % 1000, np.nan)

df_features = df_features.drop(columns=['FUNC_STAT_TCR'], errors='ignore')

In [6]:
# Select only the recommended features per data_dictionary.md
feature_cols = [
    'ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR',
    'functional_scale', 'functional_percent', 'INIT_STAT', 'INIT_CPRA',
    'INIT_AGE', 'DIALYSIS_DATE', 'INIT_DATE', 'ETHCAT', 'REGION'
]

df_model = df_features[feature_cols].copy()
df_model.head()

KeyError: "['DIALYSIS_DATE'] not in index"

In [ ]:
categorical_cols = [
    'ON_DIALYSIS', 'GENDER', 'ABO', 'A2A2B_ELIGIBILITY',
    'INIT_STAT', 'ETHCAT', 'REGION', 'functional_scale'
]

numerical_cols = ['BMI_TCR', 'INIT_CPRA', 'INIT_AGE', 'functional_percent']

# FUNC_STAT_TCR needs special decoding (per data_dictionary.md) not simple encoding or scaling
# DIALYSIS_DATE / INIT_DATE need date handling (per data_dictionary.md) not simple encoding or scaling

In [ ]:
#Caps BMI at a maximum of 80, there was a data entry error
df_model.loc[df_model['BMI_TCR'] > 80, 'BMI_TCR'] = df_model['BMI_TCR'].median()
df_model['BMI_TCR'].describe()

In [ ]:
#encode categorical data
new_df = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
new_df.head()

In [ ]:
print(categorical_cols)
for col in categorical_cols:
    print(col, df_model[col].nunique())

In [ ]:
#Check INIT_CPRA range/outliers and most frequent values (which is 0.00)
df_model['INIT_CPRA'].describe()
df_model['INIT_CPRA'].value_counts().head(10)

In [ ]:
#normalize numerical data
scaler = StandardScaler()
new_df[numerical_cols] = scaler.fit_transform(new_df[numerical_cols])
new_df[numerical_cols].head()

# Status: Task #3 mostly complete

Done: encoding + normalizing for ON_DIALYSIS, GENDER, ABO, A2A2B_ELIGIBILITY, 
INIT_STAT, ETHCAT, REGION, BMI_TCR, INIT_CPRA, INIT_AGE

Still needed:
- FUNC_STAT_TCR: needs decoding (mixed Karnofsky/Lansky scales, see data_dictionary.md)
- DIALYSIS_DATE / INIT_DATE: need conversion to a dialysis-duration feature

Errors I found: BMI_TCR had one outlier value of 430,226 (data entry error), but I it capped at 80.